# Chapter 3.4: Robot Arm Failure Prediction

Back to the course site: https://neel429.github.io/ml-for-robotics-/

**Dataset:** [AI4I 2020 Predictive Maintenance Dataset](https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020)

Run cells top to bottom. Start with Cell 1 to install the Kaggle API and paste your token.

In [ ]:
# Cell 1: Install Kaggle API & set up credentials
!pip -q install kaggle
import json
import os
from getpass import getpass

os.makedirs("/root/.kaggle", exist_ok=True)
token = json.loads(getpass("Paste your Kaggle API token text: ").strip())
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(token, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
# Cell 2: Download the dataset from Kaggle
!kaggle datasets download -d stephanmatzka/predictive-maintenance-dataset-ai4i-2020
!unzip -o predictive-maintenance-dataset-ai4i-2020.zip -d data

In [ ]:
# Cell 3: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

In [ ]:
# Cell 4: Load and explore the data
df = pd.read_csv("data/ai4i2020.csv")
display(df.head())
df.info()
display(df.describe())

In [ ]:
# Cell 5: Visualize class distribution
counts = df["Machine failure"].value_counts().sort_index()
counts.plot(kind="bar", color=["#34d399", "#fb7185"])
plt.xticks([0, 1], ["Normal", "Failure"], rotation=0)
plt.title("Failure vs normal examples")
plt.ylabel("Rows")
plt.show()

In [ ]:
# Cell 6: Visualize feature correlations
corr = df.select_dtypes(include="number").corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Numeric feature correlations")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Handle missing data
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"Total missing values: {missing.sum()}")

In [ ]:
# Cell 8: Feature selection
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
X = df[feature_cols]
y = df["Machine failure"]
print(X.shape, y.shape)

In [ ]:
# Cell 9: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
# Cell 10: Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Cell 11: Train Logistic Regression
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train_scaled, y_train)
log_pred = log_reg.predict(X_test_scaled)

In [ ]:
# Cell 12: Evaluate Logistic Regression
print(classification_report(y_test, log_pred))
ConfusionMatrixDisplay.from_predictions(y_test, log_pred, display_labels=["Normal", "Failure"])
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
# Cell 13: Train Random Forest
forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)
forest.fit(X_train, y_train)
forest_pred = forest.predict(X_test)

In [ ]:
# Cell 14: Compare both models
log_f1 = f1_score(y_test, log_pred)
forest_f1 = f1_score(y_test, forest_pred)

print(f"Logistic Regression F1: {log_f1:.3f}")
print(f"Random Forest F1:       {forest_f1:.3f}")

In [ ]:
# Cell 15: Challenge — tune n_estimators
tree_counts = [10, 25, 50, 100, 200]
scores = []

for n in tree_counts:
    model = RandomForestClassifier(n_estimators=n, random_state=42, class_weight="balanced")
    model.fit(X_train, y_train)
    scores.append(f1_score(y_test, model.predict(X_test)))

plt.plot(tree_counts, scores, marker="o")
plt.xlabel("Number of trees")
plt.ylabel("F1 score")
plt.title("Random Forest tuning")
plt.show()